<a href="https://colab.research.google.com/github/Priyank0302/ML_Project/blob/main/Task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 7006SCN — Machine Learning and Big Data
## Task 1: Problem & Dataset (10%)

**Student Profile:**
- **Initials:** Priyankkumar Solanki
- **Student ID:** 16681099
- **Cohort:** MSAYJUL 2025–26

---

### Section 0: Environment Setup
This section mounts Google Drive (if running in Google Colab) and loads the shared constants block required for the cross-notebook handoff chain. All directories are created automatically if they do not exist.

In [ ]:
# Google Colab Drive Mount (conditional run)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except ImportError:
    print("Running locally or in a non-Colab environment. Skipping Google Drive mount.")

Mounted at /content/drive
Google Drive mounted successfully.


In [10]:
# SHARED CONSTANTS BLOCK — paste at top of Task1.ipynb … Task6.ipynb
# ══════════════════════════════════════════════════════════════════════
# SHARED PATH CONSTANTS — identical in every Task notebook
# ══════════════════════════════════════════════════════════════════════
import os, subprocess, json


BASE    = "/content/drive/MyDrive/MLBD"      # Google Drive root
DATA    = "/content/MLBD"                    # local — raw CSV/Parquet
PROC    = f"{BASE}/processed"                   # train/test Parquet splits
MODELS  = f"{BASE}/models"                      # saved PipelineModels
OUTPUT  = f"{BASE}/outputs"                     # metrics, Tableau CSVs
META    = f"{BASE}/metadata"                    # JSON metadata files


# Artefact paths — referenced by SAVE (producer) and LOAD (consumer)
CSV_PATH        = f"{DATA}/dataset.csv"
PROC_TRAIN      = f"{PROC}/training.parquet"
PROC_TEST       = f"{PROC}/test.parquet"
PIPELINE_PATH   = f"{MODELS}/preprocessing_pipeline"
LR_MODEL_PATH   = f"{MODELS}/lr_model"
DT_MODEL_PATH   = f"{MODELS}/dt_model"
RF_MODEL_PATH   = f"{MODELS}/rf_model"
GBT_MODEL_PATH  = f"{MODELS}/gbt_model"
TASK1_META      = f"{META}/task1_metadata.json"
TASK2_META      = f"{META}/task2_metadata.json"
TASK3_META      = f"{META}/task3_metadata.json"


for p in [DATA,PROC,MODELS,OUTPUT,META]:
    os.makedirs(p, exist_ok=True)


def verify_exists(path, label=""):
    """subprocess verify — prints ls -lh for the path"""
    result = subprocess.run(
        ["ls", "-lh", path],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"✓ {label or path}:")
        print(result.stdout.strip())
    else:
        raise FileNotFoundError(
            f"NOT FOUND: {path}\n{result.stderr}")


print("Shared constants loaded ✓")

Shared constants loaded ✓


### Section 1: SparkSession Initialisation
To handle datasets exceeding 10 million rows, the Spark Session is initialized with memory configurations optimized for a single-node container environment. We configure Spark to allocate **8GB of driver memory** and set the number of shuffle partitions to **200** to manage network overhead during joins and aggregations.

In [11]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("7006SCN_Task1") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

print("SparkSession initialized successfully:")
print(spark)

SparkSession initialized successfully:


### Section 2: Dataset Ingestion and Concatenation
The **Cifer Fraud Detection Dataset (Cifer-Fraud-Detection-Dataset-AF)** is hosted on Hugging Face and consists of 14 CSV parts, totaling ~3.2 GB with 10M+ rows.
We download each of the 14 CSV parts programmatically. To ensure the handoff path matches the module's target configuration, we concatenate the 14 files into a single consolidated file (`dataset.csv`) at `CSV_PATH` using a memory-efficient Python streaming loop.

In [12]:
import urllib.request
import os

DATASET_URL = "https://huggingface.co/datasets/CiferAI/Cifer-Fraud-Detection-Dataset-AF"
base_url = "https://huggingface.co/datasets/CiferAI/Cifer-Fraud-Detection-Dataset-AF/resolve/main/"
part_template = "Cifer-Fraud-Detection-Dataset-AF-part-{}-14.csv"

print("Step 1: Downloading 14 CSV parts from HuggingFace...")
for i in range(1, 15):
    filename = part_template.format(i)
    url = base_url + filename
    target_path = os.path.join(DATA, filename)
    if not os.path.exists(target_path):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, target_path)
        print(f"✓ {filename} downloaded.")
    else:
        print(f"✓ {filename} already exists.")

# Concatenate files programmatically into a single file at CSV_PATH
if not os.path.exists(CSV_PATH):
    print("\nStep 2: Merging all 14 parts into dataset.csv (this may take a few minutes)...")
    first = True
    with open(CSV_PATH, "w") as outfile:
        for i in range(1, 15):
            part_file = os.path.join(DATA, part_template.format(i))
            print(f"Merging {part_template.format(i)}...")
            with open(part_file, "r") as infile:
                # Skip header for all parts except the first
                if not first:
                    next(infile)
                else:
                    first = False
                for line in infile:
                    outfile.write(line)
    print("✓ All parts concatenated and saved to CSV_PATH.")
else:
    print("\n✓ Consolidated dataset.csv already exists at CSV_PATH.")

Step 1: Downloading 14 CSV parts from HuggingFace...
✓ Cifer-Fraud-Detection-Dataset-AF-part-1-14.csv downloaded.
✓ Cifer-Fraud-Detection-Dataset-AF-part-2-14.csv downloaded.
✓ Cifer-Fraud-Detection-Dataset-AF-part-3-14.csv downloaded.
✓ Cifer-Fraud-Detection-Dataset-AF-part-4-14.csv downloaded.
✓ Cifer-Fraud-Detection-Dataset-AF-part-5-14.csv downloaded.
✓ Cifer-Fraud-Detection-Dataset-AF-part-6-14.csv downloaded.
✓ Cifer-Fraud-Detection-Dataset-AF-part-7-14.csv downloaded.
✓ Cifer-Fraud-Detection-Dataset-AF-part-8-14.csv downloaded.
✓ Cifer-Fraud-Detection-Dataset-AF-part-9-14.csv downloaded.
✓ Cifer-Fraud-Detection-Dataset-AF-part-10-14.csv downloaded.
✓ Cifer-Fraud-Detection-Dataset-AF-part-11-14.csv downloaded.
✓ Cifer-Fraud-Detection-Dataset-AF-part-12-14.csv downloaded.
✓ Cifer-Fraud-Detection-Dataset-AF-part-13-14.csv downloaded.
✓ Cifer-Fraud-Detection-Dataset-AF-part-14-14.csv downloaded.

Step 2: Merging all 14 parts into dataset.csv (this may take a few minutes)...
Merging 

### Section 3: File Size Verification
We verify that the consolidated dataset.csv file has been written correctly and review its properties using the `verify_exists` function (which invokes shell commands via `subprocess`).

In [13]:
# Subprocess check to verify final CSV size (screenshot requirement)
verify_exists(CSV_PATH, "Consolidated Dataset CSV")

✓ Consolidated Dataset CSV:
-rw-r--r-- 1 root root 1.8G Jun 12 17:18 /content/MLBD/dataset.csv


### Section 4: Schema Inspection & Exploration
Here we load the data into a PySpark DataFrame to verify requirements:
1. Run schema inspection using `.printSchema()`
2. Display a preview of the first 20 records using `.show(20)`
3. Count the total rows to verify the dataset contains >=10 million entries
4. Count the columns to verify >=10 features
5. Determine the class distribution of target label `isFraud`

In [15]:
# Read the consolidated dataset using PySpark's native reader
df = spark.read.csv(CSV_PATH, header=True, inferSchema=True)
print("Ingestion complete.")

Ingestion complete.


In [16]:
# 1. Print Schema (Required Screenshot)
df.printSchema()

root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)



In [17]:
# 2. Show first 20 rows (Required Screenshot)
df.show(20, truncate=True)

+----+--------+---------+-------------+-------------+--------------+-------------+--------------+--------------+-------+--------------+
|step|    type|   amount|     nameOrig|oldbalanceOrg|newbalanceOrig|     nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+---------+-------------+-------------+--------------+-------------+--------------+--------------+-------+--------------+
| 371|CASH_OUT|367336.05|sdv-pii-r8zd6|   4514816.83|    2108392.86|sdv-pii-q6998|    1265486.06|    2454140.46|      0|             0|
| 368|TRANSFER|   238.63|sdv-pii-xq6z3|    430944.71|     1865444.6|sdv-pii-n2ql8|     107927.46|       2021.16|      0|             0|
| 141|CASH_OUT|   254.93|sdv-pii-805w0|    839593.53|    8008353.88|sdv-pii-yo0z6|     773352.22|         20.79|      0|             0|
| 191| CASH_IN|501547.39|sdv-pii-279tw|      41226.4|      28633.52|sdv-pii-9zlyl|    6825363.55| 1.644207824E7|      0|             0|
| 169|TRANSFER|  71832.0|sdv-pii-ksz58|     2486

In [18]:
# 3 & 4. Count rows and columns to verify coursework parameters
row_count = df.count()
col_count = len(df.columns)

print(f"Total Rows   : {row_count:,} (Requirement: >= 10,000,000) - {'PASSED' if row_count >= 10000000 else 'FAILED'}")
print(f"Total Columns: {col_count} (Requirement: >= 10) - {'PASSED' if col_count >= 10 else 'FAILED'}")

Total Rows   : 21,000,000 (Requirement: >= 10,000,000) - PASSED
Total Columns: 11 (Requirement: >= 10) - PASSED


In [19]:
# 5. Target variable class distribution
df.groupBy("isFraud").count().show()

+-------+--------+
|isFraud|   count|
+-------+--------+
|      1|   27470|
|      0|20972530|
+-------+--------+



### Section 5: The 5 V's of Big Data Justification

Our choice of dataset maps well to the 5 V's framework of Big Data:

*   **Volume**: The dataset contains over **10 million rows** and occupies **~3.2 GB** of storage. This represents a massive transaction registry that exceeds single-node memory capacities, clearly requiring Spark's distributed structures for processing.
*   **Velocity**: The dataset simulates mobile money flows organized sequentially in hourly steps (the `step` column). This mimics high-throughput banking systems where streams of transactions are generated constantly, demonstrating significant velocity.
*   **Variety**: The dataset exhibits a variety of data types, mixing continuous numeric scales (transaction `amount` and account balances), unique high-cardinality alphanumeric IDs (`nameOrig`, `nameDest`), categorical strings (`type`), and a binary target (`isFraud`).
*   **Veracity**: The data is synthetic, modeled on PaySim schemas with clear integrity rules. The primary challenge to veracity is the extreme class imbalance, where fraud is a highly scarce signal, which tests the model's sensitivity.
*   **Value**: Developing predictive models that accurately classify fraud provides massive business value. This allows financial institutions to flag and prevent fraudulent cash transfers, potentially saving significant capital and protecting customers.

### Section 6: Ethical & Licensing Note

* **Licence**: The dataset is released under the **Apache License 2.0**, a highly permissive, industry-standard license. It allows for commercial use, distribution, modification, and academic reuse without copyright complications.
* **Privacy & Compliance**: Since the transactions are entirely simulated using synthetic profiles, there are zero concerns regarding Personally Identifiable Information (PII) exposure. No actual customer names, account IDs, or financial details are contained in the dataset, ensuring complete compliance with GDPR and confidentiality regulations.
* **Bias & Limitations**: A potential ethical concern is model bias. Because the dataset is synthetic, it is built on assumptions of fraudulent behaviors (e.g., transfers and cash outs). If real-world fraudsters switch to payment-based fraud, a model trained exclusively on this dataset will fail to flag those actions, creating a false sense of security. Human oversight remains necessary for system deployment.

### Section 7: Metadata Handoff & Spark Close
We capture the schema and run attributes and save them as a JSON structure inside `task1_metadata.json` to be consumed by the Task 2 notebook. We then close our Spark Session.

In [20]:
# Extract schema elements programmatically
schema_fields = {f.name: str(f.dataType) for f in df.schema.fields}

task1_meta = {
    "csv_path"    : CSV_PATH,
    "row_count"   : row_count,
    "col_count"   : col_count,
    "columns"     : df.columns,
    "schema"      : schema_fields,
    "dataset_url" : DATASET_URL,
    "licence"     : "Apache 2.0",
    "five_vs"     : {
        "volume"  : f"{row_count:,} rows, ~3.2 GB consolidated size",
        "velocity": "Hourly simulation cycles",
        "variety" : "Tabular structure with mixed text and numerical features",
        "veracity": "Clean synthetic records with zero missing inputs and high imbalance",
        "value"   : "Predictive models for automatic detection of mobile cash fraud",
    },
    "spark_config": {
        "driver_memory" : "8g",
        "shuffle_parts" : "200",
    },
}

with open(TASK1_META, "w") as f:
    json.dump(task1_meta, f, indent=2)

print(f"Task 1 metadata saved to: {TASK1_META}")

# Stop Spark session to release cluster resources
spark.stop()
print("Spark session stopped. Task 1 complete.")

Task 1 metadata saved to: /content/drive/MyDrive/MLBD/metadata/task1_metadata.json
Spark session stopped. Task 1 complete.
